# CV split audit — `stratify="both"` across the real datasets

No models, no metrics. Just inspect what the splitter actually does on each dataset
under `stratify="both"` (joint stratification on `y` and the sensitive attribute).

**Goal.** Predict — and then verify — when `stratify="both"` succeeds vs falls back
to `stratify="y"`, and look at the per-fold (y, sens) composition that results.

Uses the runner's actual private helpers (`_resolve_strat_label`, `_build_splits`),
so what we see here is exactly what `Experiment` would do.

In [1]:
import warnings
from collections import Counter

import numpy as np
import pandas as pd

from skfair.datasets import (
    load_adult, load_compas, load_german, load_heart_disease, load_ricci,
)
from skfair.experimentation._runner import _resolve_strat_label, _build_splits

# Mirrors skfair.experimentation._registry.DATASET_REGISTRY
DATASETS = [
    ('adult',         load_adult,         'sex',  1),
    ('compas',        load_compas,        'sex',  1),
    ('german',        load_german,        'sex',  1),
    ('heart_disease', load_heart_disease, 'sex',  1),
    ('ricci',         load_ricci,         'Race', 1),
]
N_SPLITS = 5
RANDOM_STATE = 42

def load(loader):
    X, y = loader()
    return X, np.asarray(y)

## 1. Pre-CV audit — joint (y, sens) cell counts

The smallest joint cell must be `≥ n_splits = 5` for full `stratify="both"` to succeed.
If it's smaller, the runner emits a `UserWarning` and falls back to `stratify="y"`.

In [2]:
audit_rows = []
joint_tables = {}

for name, loader, sens_col, priv in DATASETS:
    X, y = load(loader)
    sens = np.asarray(X[sens_col])

    joint = (
        pd.DataFrame({'y': y, sens_col: sens})
        .value_counts()
        .sort_index()
        .rename('count')
    )
    joint_tables[name] = joint.to_frame()

    audit_rows.append({
        'dataset': name,
        'n_rows': len(y),
        'y_pos_rate': float((y == 1).mean()),
        'sens_priv_rate': float((sens == priv).mean()),
        'n_joint_cells': len(joint),
        'smallest_joint_cell': int(joint.min()),
        'predicted_fallback': int(joint.min()) < N_SPLITS,
    })

audit = pd.DataFrame(audit_rows).set_index('dataset')
audit.round(3)

,n_rows,y_pos_rate,sens_priv_rate,n_joint_cells,smallest_joint_cell,predicted_fallback
dataset,,,,,,
adult,32561,0.241,0.669,4,1179,False
compas,7214,0.451,0.807,4,498,False
german,1000,0.700,0.690,4,109,False
heart_disease,270,0.556,0.678,4,20,False
ricci,118,0.475,0.576,4,15,False


In [3]:
# Joint cell tables, side by side
for name, df in joint_tables.items():
    print(f'=== {name} ===')
    display(df)

=== adult ===


count
y sex       
0 0     9592
  1    15128
1 0     1179
  1     6662

=== compas ===


count
y sex       
0 0      897
  1     3066
1 0      498
  1     2753

=== german ===


count
y sex       
0 0      109
  1      191
1 0      201
  1      499

=== heart_disease ===


count
y sex       
0 0       20
  1      100
1 0       67
  1       83

=== ricci ===


count
y Race       
0 0        35
  1        27
1 0        15
  1        41

## 2. Run the actual splitter — does fallback trigger?

Call `_build_splits` with `stratify="both"` and capture any warnings. The runner
warns with `"falling back to stratify='y'"` whenever it has to degrade.

In [4]:
splits_per_dataset = {}
fallback_rows = []

for name, loader, sens_col, priv in DATASETS:
    X, y = load(loader)
    y_arr = np.asarray(y)

    # Resolver: 'both' → joint string label of (y, sens)
    label = _resolve_strat_label('both', y_arr, X, sens_col)

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        splits = _build_splits(
            n_splits=N_SPLITS, n_repeats=1,
            strat_label=label, y_arr=y_arr, random_state=RANDOM_STATE,
        )
        fallback = any('falling back' in str(w.message) for w in caught)
        warning_text = '; '.join(str(w.message) for w in caught) or '—'

    splits_per_dataset[name] = (splits, label, fallback)
    fallback_rows.append({
        'dataset':           name,
        'n_folds_obtained':  len(splits),
        'fallback_to_y':     fallback,
        'warning':           warning_text,
    })

pd.DataFrame(fallback_rows).set_index('dataset')

,n_folds_obtained,fallback_to_y,warning
dataset,,,
adult,5,False,—
compas,5,False,—
german,5,False,—
heart_disease,5,False,—
ricci,5,False,—


## 3. Per-fold composition (test sets)

For each dataset, count the (y, sens) cells in each fold's **test** indices.
A well-stratified fold has every cell represented at roughly equal counts.
If fallback to `y` happened, the group counts are no longer guaranteed to be
balanced fold-to-fold.

In [5]:
for name, loader, sens_col, priv in DATASETS:
    X, y = load(loader)
    y_arr = np.asarray(y)
    sens = np.asarray(X[sens_col])

    splits, label, fallback = splits_per_dataset[name]

    rows = []
    for _, test_idx in splits:
        cnt = Counter(zip(y_arr[test_idx], sens[test_idx]))
        rows.append({
            **{f'(y={k[0]}, {sens_col}={k[1]})': cnt[k] for k in sorted(cnt)},
            'fold_size': len(test_idx),
        })
    df = pd.DataFrame(rows).fillna(0).astype(int)
    df.index.name = (
        f'fold ({"FELL BACK to stratify=y" if fallback else "stratify=both succeeded"})'
    )
    print(f'=== {name} ===')
    display(df)

=== adult ===


,"(y=0, sex=0)","(y=0, sex=1)","(y=1, sex=0)","(y=1, sex=1)",fold_size
fold (stratify=both succeeded),,,,,
0,1918,3026,236,1333,6513
1,1918,3026,235,1333,6512
2,1918,3026,236,1332,6512
3,1919,3025,236,1332,6512
4,1919,3025,236,1332,6512


=== compas ===


,"(y=0, sex=0)","(y=0, sex=1)","(y=1, sex=0)","(y=1, sex=1)",fold_size
fold (stratify=both succeeded),,,,,
0,180,614,99,550,1443
1,179,613,100,551,1443
2,179,613,100,551,1443
3,179,613,100,551,1443
4,180,613,99,550,1442


=== german ===


,"(y=0, sex=0)","(y=0, sex=1)","(y=1, sex=0)","(y=1, sex=1)",fold_size
fold (stratify=both succeeded),,,,,
0,22,38,40,100,200
1,22,38,40,100,200
2,22,38,40,100,200
3,21,39,40,100,200
4,22,38,41,99,200


=== heart_disease ===


,"(y=0, sex=0)","(y=0, sex=1)","(y=1, sex=0)","(y=1, sex=1)",fold_size
fold (stratify=both succeeded),,,,,
0,4,20,14,16,54
1,4,20,14,16,54
2,4,20,13,17,54
3,4,20,13,17,54
4,4,20,13,17,54


=== ricci ===


,"(y=0, Race=0)","(y=0, Race=1)","(y=1, Race=0)","(y=1, Race=1)",fold_size
fold (stratify=both succeeded),,,,,
0,7,5,3,9,24
1,7,6,3,8,24
2,7,6,3,8,24
3,7,5,3,8,23
4,7,5,3,8,23


## 4. Summary

In [6]:
summary = audit.join(
    pd.DataFrame(fallback_rows).set_index('dataset')[['fallback_to_y']]
)
summary['prediction_correct'] = (
    summary['predicted_fallback'] == summary['fallback_to_y']
)
summary

,n_rows,y_pos_rate,sens_priv_rate,n_joint_cells,smallest_joint_cell,predicted_fallback,fallback_to_y,prediction_correct
dataset,,,,,,,,
adult,32561,0.240810,0.669205,4,1179,False,False,True
compas,7214,0.450652,0.806626,4,498,False,False,True
german,1000,0.700000,0.690000,4,109,False,False,True
heart_disease,270,0.555556,0.677778,4,20,False,False,True
ricci,118,0.474576,0.576271,4,15,False,False,True


## Takeaways

**`stratify="both"` works cleanly on all 5 real datasets — no fallback.** Every dataset produced 5 full folds with the joint `(y, sens)` strat label; no `UserWarning` was emitted; `predicted_fallback` matched `fallback_to_y` (both `False`) for all five.

**Smallest joint cell is 15 (`ricci`)** — three times the `n_splits=5` threshold. The next smallest is `heart_disease` at 20. Both are comfortably above the fallback boundary.

**Per-fold composition is well-balanced.** Even on `ricci` (118 rows, smallest dataset), every fold's test set contains ≥3 of every `(y, Race)` cell. No fold is missing a cell, on any dataset. Joint stratification is doing what it's supposed to.

**Implication for the y vs both comparison.** The systematic fairness-metric drift we saw in `compare_stratification.ipynb` is **not** a fallback artifact — both strategies produce valid 5-fold splits on every dataset. The drift comes from genuinely different fold compositions: `y`-only strat lets the per-fold group balance vary by chance, `both` enforces it. So the differences observed there are real consequences of stratification choice, not a degenerate-fold edge case.

**Practical recommendation.** For these 5 datasets, `stratify="both"` is safe to use at `n_splits=5`. A future dataset with a smaller minority intersection (joint cell < 5) is the only thing to watch for; the runner will warn and fall back automatically.